<a href="https://colab.research.google.com/github/joseportocarrero-stack/Tinylittlescripts/blob/main/CanvasAutomationTaskProject_v1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The purpose of this script is the automation of homework and exam search for my courses enrolled in Tecsup.

In [7]:
import os
from canvasapi import Canvas
from datetime import datetime, timezone
import zoneinfo

# Try to get secrets from Colab first; fall back to environment variables
try:
    !pip install canvasapi
    from google.colab import userdata
    API_URL = userdata.get('CANVAS_URL')
    API_KEY = userdata.get('CANVAS_KEY')
except ImportError:
    # Not in Colab – use environment variables (GitHub Actions)
    API_URL = os.getenv("CANVAS_URL")
    API_KEY = os.getenv("CANVAS_KEY")

canvas = Canvas(API_URL, API_KEY)
LOCAL_TZ = zoneinfo.ZoneInfo("America/Lima")   # GMT-5

In [8]:
def get_tasks():
    print("Conecting a Canvas...\n")
    user = canvas.get_current_user()
    courses = user.get_courses(enrollment_state="active")
    report = []

    now_local = datetime.now(LOCAL_TZ)
    start_of_today = now_local.replace(hour=0, minute=0, second=0, microsecond=0)

    for course in courses:
        # Old courses
        if not hasattr(course, 'name'):
            continue
        print(f"Analyzing: {course.name}...")

        # Obtaining tasks
        try:
            tasks = course.get_assignments()
            for task in tasks:
              if task.due_at:
                # Parse UTC timestamp and convert to local time
                deadline_utc = datetime.strptime(
                    task.due_at, "%Y-%m-%dT%H:%M:%SZ"
                ).replace(tzinfo=timezone.utc)
                deadline_local = deadline_utc.astimezone(LOCAL_TZ)
                # Filter: Only show tasks that are due as of today
                if deadline_local >= start_of_today:
                  report.append({
                      "Course": course.name,
                      "Type": "Task",
                      "Name": task.name,
                      "Deadline": deadline_local.strftime("%Y-%m-%d %H:%M GMT-5"),
                      "Link": task.html_url
                  })
        except Exception as e:
            # Sometimes there are no permissions or the course does not have the module active
            pass

    lines = []
    lines.append("="*50)
    lines.append("📋 REPORT OF PENDING ASSIGNMENTS AND EXAMS")
    lines.append("="*50)
    if not report:
        lines.append("✅ Up to date! No pending tasks.")
    else:
        for item in report:
            lines.append(f"\n📚 <b>{item['Course']}</b>")
            lines.append(f"📌 {item['Type']}: {item['Name']}")
            lines.append(f"📅 Deadline: {item['Deadline']} (GMT-5)")
            # Use HTML link so it’s clickable in Telegram
            lines.append(f'🔗 <a href="{item["Link"]}">Open in Canvas</a>')
            lines.append("-" * 30)
    return "\n".join(lines)

In [9]:
if __name__ == "__main__":
    report = get_tasks()
    print(report)   # always show in Actions log

    # Send via Telegram (optional – fails gracefully if credentials missing)
    try:
        from telegram_sender import send_telegram_message
        send_telegram_message(report)
    except ImportError:
        print("telegram_sender module not found – cannot send Telegram message.")

Conecting a Canvas...

Analyzing: Automatización de Procesos de Datos con IA - C28R 1ero A-L - C28R 1ero B-L-L...
Analyzing: Cálculo y Estadística - C28R 1ero A-L - C28R 1ero B-L-L...
Analyzing: Curso de Inducción 2026-1...
Analyzing: Desarrollo Personal - C28R 1ero A-L - C28R 1ero B-L-L...
Analyzing: Fundamentos de Gestión de Datos - C28R 1ero A-L - C28R 1ero B-L-L...
Analyzing: Introducción a los Sistemas Informáticos y sus Aplicaciones - C28R 1ero A-L - C28R 1ero B-L-L...
Analyzing: Programación para Ciencia de Datos - C28R 1ero A-L - C28R 1ero B-L-L...
Analyzing: Técnicas de Expresión Oral y Escrita - C28R 1ero A-L - C28R 1ero B-L-L...
Analyzing: Tutoría 1 - C28R 1ero A-L - C28R 1ero B-L-L...
📋 REPORT OF PENDING ASSIGNMENTS AND EXAMS

📚 <b>Desarrollo Personal - C28R 1ero A-L - C28R 1ero B-L-L</b>
📌 Task: Actividad de desarrollo de la semana 10 | Caso de estudio: “La broma sin gracia”
📅 Deadline: 2026-07-18 23:59 GMT-5 (GMT-5)
🔗 <a href="https://tecsup.instructure.com/courses/69396/